In [3]:
import os
import cv2
import numpy as np

# 1. Carregamento do vídeo
caminho_video = "videos/video_3.mp4"
if not os.path.exists(caminho_video):
    raise FileNotFoundError(f"Vídeo '{caminho_video}' não encontrado.")

cap = cv2.VideoCapture(caminho_video)
largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

caminho_saida = "resultado_bifurcacao_14_7.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(caminho_saida, fourcc, fps, (largura, altura))

# 2. Captura do quadro inicial (vazio) como referência absoluta de fundo
# Os primeiros frames (0.0s a 1.0s) não possuem caixas nas zonas de saída
sucesso, frame_base = cap.read()
if not sucesso:
    raise RuntimeError("Não foi possível ler o primeiro frame.")

fundo_cinza = cv2.cvtColor(frame_base, cv2.COLOR_BGR2GRAY)
fundo_cinza = cv2.GaussianBlur(fundo_cinza, (5, 5), 0)

# Volta o vídeo para o início
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

# 3. Definição das Zonas de Sensores Virtuais (ROIs após a bifurcação)
# Sensor 1: Entrada da Rampa Metálica (Direção Esquerda -> 14 caixas)
s1_x1, s1_y1 = int(largura * 0.35), int(altura * 0.25)
s1_x2, s1_y2 = int(largura * 0.46), int(altura * 0.41)

# Sensor 2: Saída da Esteira Azul/Roletes (Direção Direita -> 7 caixas)
s2_x1, s2_y1 = int(largura * 0.72), int(altura * 0.26)
s2_x2, s2_y2 = int(largura * 0.90), int(altura * 0.44)

# Variáveis de Estado e Contadores
contagem_esquerda = 0
ocupado_esq = False
cooldown_esq = 0

contagem_direita = 0
ocupado_dir = False
cooldown_dir = 0

print("Processando bifurcação das esteiras...")

while cap.isOpened():
    sucesso, frame = cap.read()
    if not sucesso:
        break

    atual_cinza = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    atual_cinza = cv2.GaussianBlur(atual_cinza, (5, 5), 0)

    # Diferença absoluta em relação à esteira vazia + filtro de brilho das caixas
    diff = cv2.absdiff(atual_cinza, fundo_cinza)
    _, mascara_diff = cv2.threshold(diff, 28, 255, cv2.THRESH_BINARY)

    # =========================================================================
    # AVALIAÇÃO DO SENSOR 1: RAMPA ESQUERDA (14 CAIXAS)
    # =========================================================================
    roi_esq = mascara_diff[s1_y1:s1_y2, s1_x1:s1_x2]
    taxa_esq = (cv2.countNonZero(roi_esq) / roi_esq.size) * 100.0

    if cooldown_esq > 0:
        cooldown_esq -= 1

    if not ocupado_esq:
        if taxa_esq > 22.0 and cooldown_esq == 0:
            ocupado_esq = True
            contagem_esquerda += 1
            cooldown_esq = int(fps * 0.35)  # Evita recontagem da mesma caixa
    else:
        if taxa_esq < 10.0:
            ocupado_esq = False

    # =========================================================================
    # AVALIAÇÃO DO SENSOR 2: ESTEIRA DIREITA (7 CAIXAS)
    # =========================================================================
    roi_dir = mascara_diff[s2_y1:s2_y2, s2_x1:s2_x2]
    taxa_dir = (cv2.countNonZero(roi_dir) / roi_dir.size) * 100.0

    if cooldown_dir > 0:
        cooldown_dir -= 1

    if not ocupado_dir:
        if taxa_dir > 20.0 and cooldown_dir == 0:
            ocupado_dir = True
            contagem_direita += 1
            cooldown_dir = int(fps * 0.45)  # Caixas à direita passam mais devagar
    else:
        if taxa_dir < 8.0:
            ocupado_dir = False

    # =========================================================================
    # INTERFACE VISUAL E AUDITORIA NO VÍDEO
    # =========================================================================
    cor_s1 = (0, 255, 0) if ocupado_esq else (0, 165, 255)
    cor_s2 = (0, 255, 0) if ocupado_dir else (255, 180, 0)

    # Desenha Sensor 1 (Esquerda)
    cv2.rectangle(frame, (s1_x1, s1_y1), (s1_x2, s1_y2), cor_s1, 2)
    cv2.putText(frame, f"Dir 1 (Rampa): {contagem_esquerda}", (s1_x1 - 20, s1_y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, cor_s1, 2)

    # Desenha Sensor 2 (Direita)
    cv2.rectangle(frame, (s2_x1, s2_y1), (s2_x2, s2_y2), cor_s2, 2)
    cv2.putText(frame, f"Dir 2 (Reto): {contagem_direita}", (s2_x1 - 10, s2_y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, cor_s2, 2)

    # Painel Consolidado no Topo
    cv2.rectangle(frame, (15, 15), (430, 125), (0, 0, 0), -1)
    cv2.putText(frame, f"DIRECAO 1 (RAMPA ESQ): {contagem_esquerda}", (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 0), 2)
    cv2.putText(frame, f"DIRECAO 2 (ESTEIRA DIR): {contagem_direita}", (30, 85),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 200, 0), 2)
    cv2.putText(frame, f"TOTAL GERAL: {contagem_esquerda + contagem_direita}", (30, 115),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 1)

    out.write(frame)

cap.release()
out.release()

print(f"\n--- RESULTADO FINAL ---")
print(f"Direção 1 (Rampa Esquerda): {contagem_esquerda} caixas")
print(f"Direção 2 (Esteira Direita): {contagem_direita} caixas")
print(f"Total: {contagem_esquerda + contagem_direita} caixas")
print(f"Vídeo salvo em: {os.path.abspath(caminho_saida)}")

Processando bifurcação das esteiras...

--- RESULTADO FINAL ---
Direção 1 (Rampa Esquerda): 14 caixas
Direção 2 (Esteira Direita): 7 caixas
Total: 21 caixas
Vídeo salvo em: C:\Users\fabri\Documents\pythonScripts\Aula 24\resultado_bifurcacao_14_7.mp4
